# Sentiment Analysis
**Nama**: Asep Haryana Saputra

## 1. Import Libraries

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import tensorflow as tf
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
try:
    from tensorflow.keras.layers import TextVectorization, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
except ImportError:
    from tensorflow.keras.layers.experimental.preprocessing import TextVectorization
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional, GRU, Dropout, GlobalAveragePooling1D, Input, SpatialDropout1D, BatchNormalization, Conv1D, GlobalMaxPooling1D
from tensorflow.keras.models import Sequential, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")

# Optimize GPU Memory
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)


TensorFlow Version: 2.20.0


In [12]:
# Callbacks Definition
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

def get_callbacks(model_name):
    checkpoint = ModelCheckpoint(
        f'{model_name}_best.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        min_delta=0.001,
        patience=10,
        restore_best_weights=True,
        verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
    return [checkpoint, early_stopping, reduce_lr]

## 2. Data Loading & Preprocessing
**Labeling Strategy**:
- 1-2 Stars -> **Negative (0)**
- 3 Stars -> **Neutral (1)**
- 4-5 Stars -> **Positive (2)**

In [13]:
# Nama file dataset
filename = 'dataset_raw.csv'

# Daftar kemungkinan lokasi file (Relative Path)
possible_paths = [
    f'./{filename}',                      # 1. Di folder yang sama (Ideal)
    f'./Analisis_Sentimen/{filename}',    # 2. Jika notebook dijalankan dari root project
    f'../{filename}',                     # 3. Jika notebook ada di subfolder
    f'/content/drive/MyDrive/{filename}'  # 4. Jika di Google Colab (opsional)
]

dataset_path = None
for path in possible_paths:
    if os.path.exists(path):
        dataset_path = path
        print(f"File ditemukan di: {dataset_path}")
        break

if dataset_path is None:
    raise FileNotFoundError(f"Gagal menemukan {filename}. Pastikan file ada di folder yang sama dengan notebook.")

# Load dataset dengan path yang sudah ditemukan
df = pd.read_csv(dataset_path)

# Create 'label' column based on 'score'
def get_sentiment_label(score):
    if score <= 2:
        return 0  # Negative
    elif score == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['score'].apply(get_sentiment_label)
df.head()

File ditemukan di: ./dataset_raw.csv


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label
0,bb147f3b-ece2-494c-b6a4-cec369082e09,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,Fantastic game! Been playing for two years!,5,0,NaN,2026-02-07 09:11:59,Happy to hear you've been here for so long!,2026-02-09 22:45:30,NaN,2
1,c94054e7-6746-4d26-a1b8-f6e6d9b0be0b,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,like Pokemon go meets FF fun stuff,5,0,3.20.5\n,2026-02-07 02:24:03,Thanks for the review!,2026-02-09 22:44:46,3.20.5\n,2
2,da12f918-2bef-4395-8cc9-8a2810003377,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,this is my speed so far,5,0,3.20.5\n,2026-02-05 08:38:23,NaN,NaN,3.20.5\n,2
3,58ca197f-3a67-42a9-bde5-fc7d12a3a73b,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,boring,1,0,3.20.5\n,2026-02-04 09:10:52,Thanks for giving Orna a try!,2026-02-04 23:00:53,3.20.5\n,0
4,2943c065-2697-4b1c-821e-f8132ab71785,A Google user,https://play-lh.googleusercontent.com/EGemoI2N...,excellent,5,0,3.20.5\n,2026-02-02 08:24:57,Thanks for the review!,2026-02-03 04:44:55,3.20.5\n,2


In [14]:
# Advanced Data Cleaning (Lemmatization + POS Tagging + Emoji Preservation)
import nltk
import emoji
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download necessary NLTK data
nltk_packages = ['stopwords', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'omw-1.4', 'punkt']
for pkg in nltk_packages:
    try:
        if pkg == 'punkt':
             nltk.data.find('tokenizers/punkt')
        elif pkg in ['stopwords', 'wordnet', 'omw-1.4']:
             nltk.data.find(f'corpora/{pkg}')
        else:
             nltk.data.find(f'taggers/{pkg}')
    except LookupError:
        print('downloading')
        nltk.download(pkg)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Explicitly keep negation words AND conjunctions like "but" for nuance
negation_words = {'no', 'not', 'nor', 'neither', 'none', 'never', "don't", "can't", "won't", "isn't", "aren't"}
context_words = {'but', 'however', 'although'} 
stop_words = stop_words - negation_words - context_words

# Contraction Map (Standard English)
contraction_map = {
    "don't": "do not", "can't": "cannot", "won't": "will not",
    "isn't": "is not", "aren't": "are not", "didn't": "did not",
    "doesn't": "does not", "haven't": "have not", "hadn't": "had not",
    "wouldn't": "would not", "couldn't": "could not", "shouldn't": "should not",
    "it's": "it is", "i'm": "i am", "you're": "you are", "we're": "we are", "they're": "they are"
}

def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    elif tag.startswith('V'): return wordnet.VERB
    elif tag.startswith('R'): return wordnet.ADV
    else: return wordnet.NOUN

def clean_text(text):
    text = str(text)
    
    # 0. Demojize (Convert emojis to text: 👍 -> :thumbs_up:)
    # We replace spaces with underscores to keep them as single tokens if needed, 
    # but 'thumbs up' as separate tokens is also fine. 
    # Let's use loose conversion to keep English words.
    text = emoji.demojize(text, delimiters=(" ", " ")) 
    text = text.replace("_", " ") # thumbs_up -> thumbs up (better for embeddings)
    
    text = text.lower()
    
    # 1. Expand Basic Contractions
    for word, replacement in contraction_map.items():
        text = text.replace(word, replacement)
    
    # 2. Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # 3. Simple Regex (Keep alphanumeric + basic punctuation)
    # Note: We must NOT remove the text generated from emojis!
    # Our regex [^a-zA-Z0-9\s] preserves text, so 'thumbs up' is safe.
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text) 
    
    # 4. Tokenization (Simple Split for speed/robustness)
    tokens = text.split()
    if not tokens: return ""
    
    pos_tags = nltk.pos_tag(tokens)
    
    # 5. Lemmatization with POS & Stopword Removal
    cleaned_tokens = []
    for word, tag in pos_tags:
        if word not in stop_words:
            # Lemmatize
            lemma = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            cleaned_tokens.append(lemma)
            
    return " ".join(cleaned_tokens)

# Test the cleaner
print("Original: Clean logic 👍 but slow 😭")
print("Cleaned :", clean_text("Clean logic 👍 but slow 😭"))



downloading
downloading
Original: Clean logic 👍 but slow 😭
Cleaned : clean logic thumb but slow loudly cry face


[nltk_data] Downloading package wordnet to /home/asephs/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/asephs/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [15]:
from tqdm.auto import tqdm
tqdm.pandas()

# Apply cleaning with progress bar
if 'content' in df.columns:
    print("Cleaning text... params: lemmatization=True, emojis=True")
    df['cleaned_content'] = df['content'].progress_apply(clean_text)
    print("Cleaning complete.")


Cleaning text... params: lemmatization=True, emojis=True


  0%|          | 0/9667 [00:00<?, ?it/s]

Cleaning complete.


In [16]:
# 3. Data Quality Filtering & Splitting
if 'content' in df.columns:
    # 1. Remove Timestamp Rows
    timestamp_pattern = r'^\\d{2}:\\d{2}:\\d{2}$'
    df_clean = df[~df['content'].astype(str).str.match(timestamp_pattern)].copy()
    
    # 2. Remove Short Rows (< 5 chars) which are not Emojis?
    # Actually, now that we preserve emojis, a single emoji might be meaningful.
    # But user agreed to drop < 5. "👍" becomes " thumbs up " (9 chars).
    # So we should apply short text filter AFTER demojizing or assume raw < 5 is trash.
    # Let's stick to user request: drop length < 5 raw.
    df_clean = df_clean[df_clean['content'].astype(str).str.len() >= 5].copy()
    
    # 3. Sample Weighting (Log-based)
    # Weight = log(thumbsUp + 1) + 1 (Base weight 1)
    # We add 1 to ensure minimum weight is 1.
    # Or just log1p(thumbsUp) + 1.
    # If thumbsUp=0, weight=1. If thumbsUp=100, weight=~5.6.
    import numpy as np
    df_clean['sample_weight'] = np.log1p(df_clean['thumbsUpCount']) + 1.0
    
    print(f"Original shape: {df.shape}, Cleaned shape: {df_clean.shape}")

    X = df_clean['content'].values
    y = df_clean['label'].values
    weights = df_clean['sample_weight'].values

    # Stratified Split (80/20) - Carrying weights!
    # train_test_split doesn't split weights automatically unless we zip them.
    # We need to split indices to keep weights mapped correctly.
    indices = np.arange(len(df_clean))
    
    idx_train_80, idx_test_80, y_train_80, y_test_80 = train_test_split(indices, y, test_size=0.2, random_state=42, stratify=y)
    
    X_train_80 = X[idx_train_80]
    X_test_80 = X[idx_test_80]
    w_train_80 = weights[idx_train_80]
    # w_test_80 = weights[idx_test_80] # Validation weights usually 1, or matters less

    # Scheme 2: 90/10 Split
    idx_train_90, idx_test_90, y_train_90, y_test_90 = train_test_split(indices, y, test_size=0.1, random_state=42, stratify=y)
    
    X_train_90 = X[idx_train_90]
    w_train_90 = weights[idx_train_90]
    X_test_90 = X[idx_test_90] # Just for shape printing

    print(f"Scheme 1/3 (80/20) - Train: {X_train_80.shape}, Weights: {w_train_80.shape}")
    print(f"Scheme 2 (90/10)   - Train: {X_train_90.shape}, Weights: {w_train_90.shape}")
else:
    print("Error: 'content' column not found.")



Original shape: (9667, 13), Cleaned shape: (9491, 14)
Scheme 1/3 (80/20) - Train: (7592,), Weights: (7592,)
Scheme 2 (90/10)   - Train: (8541,), Weights: (8541,)


## 3. Tokenization & Splitting

In [17]:
# Tokenization for Scheme 1 & 2 (Embedding Models)
VOCAB_SIZE = 10000
MAX_LENGTH = 100
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

# Prepare Sequences for Scheme 1 (80/20)
seq_train_80 = tokenizer.texts_to_sequences(X_train_80)
seq_test_80 = tokenizer.texts_to_sequences(X_test_80)
pad_train_80 = pad_sequences(seq_train_80, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_80 = pad_sequences(seq_test_80, maxlen=MAX_LENGTH, padding='post', truncating='post')

# Prepare Sequences for Scheme 2 (90/10)
seq_train_90 = tokenizer.texts_to_sequences(X_train_90)
seq_test_90 = tokenizer.texts_to_sequences(X_test_90)
pad_train_90 = pad_sequences(seq_train_90, maxlen=MAX_LENGTH, padding='post', truncating='post')
pad_test_90 = pad_sequences(seq_test_90, maxlen=MAX_LENGTH, padding='post', truncating='post')

In [18]:
# Force CPU for TextVectorization to avoid GPU OOM
with tf.device('/CPU:0'):
    # Preprocessing for Scheme 3 (TF-IDF)
    vectorize_layer = TextVectorization(
        max_tokens=VOCAB_SIZE,
        output_mode='tf-idf',
        ngrams=2 # improved features
    )
    # Adapt on 80% train data
    vectorize_layer.adapt(X_train_80)

    tfidf_train_80 = vectorize_layer(X_train_80)
    tfidf_test_80 = vectorize_layer(X_test_80)


## 3.1 Load Pre-trained Embeddings


## 4. Experiment 1: Bi-LSTM Model

In [19]:
# Scheme 1 (Baseline): Bi-LSTM + Embedding + 80/20 Split
tf.random.set_seed(42)
model1 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme1_BiLSTM')

model1.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model1.summary()

history1 = model1.fit(
    pad_train_80, y_train_80,
    epochs=10,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme1_bilstm')
)


/home/asephs/venv/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "Scheme1_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
237/238 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6268 - loss: 0.9285
Epoch 1: val_accuracy improved from None to 0.80990, saving model to scheme1_bilstm_best.keras

Epoch 1: finished saving model to scheme1_bilstm_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - accuracy: 0.7467 - loss: 0.7355 - val_accuracy: 0.8099 - val_loss: 0.6719 - learning_rate: 0.0010
Epoch 2/10
237/238 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8413 - loss: 0.5000
Epoch 2: val_accuracy improved from 0.80990 to 0.81464, saving model to scheme1_bilstm_best.keras

Epoch 2: finished saving model to scheme1_bilstm_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8516 - loss: 0.4715 - val_accuracy: 0.8146 - val_loss: 0.6492 - learning_rate: 0.0010
Epoch 3/10
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8754 - loss: 0.3901
Epoch 3: val_accuracy improved from 0.81464 to 0.82412, saving model to scheme1_bilstm_best.keras

Epoch 3: finished saving model to scheme1

## 5. Experiment 2: GRU Model

In [20]:
# Scheme 2 (Data Variation): GRU + Embedding + 90/10 Split
tf.random.set_seed(42)
model2 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    GRU(128, return_sequences=True),
    Dropout(0.3),
    GRU(64),
    Dropout(0.3),
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme2_GRU_90_10')

model2.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model2.summary()

history2 = model2.fit(
    pad_train_90, y_train_90,
    epochs=10,
    validation_data=(pad_test_90, y_test_90),
    batch_size=32,
    callbacks=get_callbacks('scheme2_gru')
)


Model: "Scheme2_GRU_90_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_2             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
263/267 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6187 - loss: 0.9543
Epoch 1: val_accuracy improved from None to 0.80947, saving model to scheme2_gru_best.keras

Epoch 1: finished saving model to scheme2_gru_best.keras
267/267 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.7417 - loss: 0.8020 - val_accuracy: 0.8095 - val_loss: 0.6624 - learning_rate: 0.0010
Epoch 2/10
261/267 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8103 - loss: 0.6697
Epoch 2: val_accuracy did not improve from 0.80947
267/267 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8096 - loss: 0.6698 - val_accuracy: 0.8095 - val_loss: 0.6538 - learning_rate: 0.0010
Epoch 3/10
267/267 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8104 - loss: 0.6534
Epoch 3: val_accuracy did not improve from 0.80947
267/267 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8097 - loss: 0.6548 - val_accuracy: 0.8095 - val_loss: 0.6426 - learning_rate: 0.0010
Epoch 4/10
262/267 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8

## 6. Experiment 3: Simple NN (Baseline)

In [21]:
# Scheme 3 (Feature Variation): Dense + TF-IDF + 80/20 Split
tf.random.set_seed(42)
model3 = Sequential([
    Input(shape=(VOCAB_SIZE,)), # TF-IDF input
    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(3, activation='softmax')
], name='Scheme3_Dense_TFIDF')

model3.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model3.summary()

history3 = model3.fit(
    tfidf_train_80, y_train_80,
    epochs=10,
    validation_data=(tfidf_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme3_tfidf')
)


Model: "Scheme3_Dense_TFIDF"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 256)            │     2,560,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,593,539 (9.89 MB)

 Trainable params: 2,593,539 (9.89 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


... (2 outputs left out)

I0000 00:00:1770711109.395825  569272 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7462 - loss: 1.2844

2026-02-10 15:11:51.696912: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-02-10 15:11:51.927306: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47', 8 bytes spill stores, 8 bytes spill loads

2026-02-10 15:11:53.190249: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_47', 8 bytes spill stores, 8 bytes spill loads




Epoch 1: val_accuracy improved from None to 0.83728, saving model to scheme3_tfidf_best.keras

Epoch 1: finished saving model to scheme3_tfidf_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.7903 - loss: 0.9456 - val_accuracy: 0.8373 - val_loss: 0.7384 - learning_rate: 0.0010
Epoch 2/10
227/238 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8437 - loss: 0.7115
Epoch 2: val_accuracy improved from 0.83728 to 0.84781, saving model to scheme3_tfidf_best.keras

Epoch 2: finished saving model to scheme3_tfidf_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8521 - loss: 0.6885 - val_accuracy: 0.8478 - val_loss: 0.7457 - learning_rate: 0.0010
Epoch 3/10
234/238 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8976 - loss: 0.6023
Epoch 3: val_accuracy did not improve from 0.84781
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9007 - loss: 0.5854 - val_accuracy: 0.8399 - val_loss: 0.7964 - learning_rate: 0.0010
Epoch 4/10
215/238 ━━━━━━━━━━━━━━━━━━━━ 0

## 8. Experiment 5: Conv1D (N-Gram Learning)


In [22]:
# Scheme 5: Conv1D for N-Gram feature extraction
# Conv1D filters act as N-gram detectors (e.g. kernel_size=2 sees 2 words)
tf.random.set_seed(42)
model5 = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LENGTH),
    SpatialDropout1D(0.3),
    Conv1D(128, 5, activation='relu'), # Detects 5-grams
    GlobalMaxPooling1D(), # Captures most important features
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax')
], name='Scheme5_Conv1D_NGram')

model5.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model5.summary()

history5 = model5.fit(
    pad_train_80, y_train_80,
    epochs=50,
    validation_data=(pad_test_80, y_test_80),
    batch_size=32,
    callbacks=get_callbacks('scheme5_conv1d')
)


Model: "Scheme5_Conv1D_NGram"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_3             │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


... (2 outputs left out)

2026-02-10 15:12:03.576750: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.70GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-02-10 15:12:03.677513: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.60GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6099 - loss: 0.9570

2026-02-10 15:12:05.139193: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 2.00GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2026-02-10 15:12:05.826034: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.74GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.



Epoch 1: val_accuracy improved from None to 0.80990, saving model to scheme5_conv1d_best.keras

Epoch 1: finished saving model to scheme5_conv1d_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.7306 - loss: 0.7530 - val_accuracy: 0.8099 - val_loss: 0.6683 - learning_rate: 0.0010
Epoch 2/50
235/238 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8576 - loss: 0.4531
Epoch 2: val_accuracy improved from 0.80990 to 0.81095, saving model to scheme5_conv1d_best.keras

Epoch 2: finished saving model to scheme5_conv1d_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8658 - loss: 0.4234 - val_accuracy: 0.8110 - val_loss: 0.6277 - learning_rate: 0.0010
Epoch 3/50
220/238 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9067 - loss: 0.3083
Epoch 3: val_accuracy improved from 0.81095 to 0.83096, saving model to scheme5_conv1d_best.keras

Epoch 3: finished saving model to scheme5_conv1d_best.keras
238/238 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9182 - loss: 0.

## 7. Evaluation & Inference

In [23]:
# Inference Function
def predict_sentiment(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    # Use explicit padding/truncating to match training (post)
    padded = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post', truncating='post')
    pred = model1.predict(padded)
    label_idx = np.argmax(pred)

    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    return mapping[label_idx]

# Test - multiple examples (includes several neutral examples)
examples = [
    "This game is super fun!",
    "I love the new biomes; it's really immersive.",
    "It's okay, not great but not bad.",
    "The update added content; I don't have a strong opinion.",
    "Average performance — runs fine for me.",
    "Too many bugs, unplayable.",
    "Worked perfectly, no complaints.",
    "Controls are confusing sometimes."
]

for s in examples:
    print(s, "->", predict_sentiment(s))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step
This game is super fun! -> Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
I love the new biomes; it's really immersive. -> Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
It's okay, not great but not bad. -> Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
The update added content; I don't have a strong opinion. -> Neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
Average performance — runs fine for me. -> Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Too many bugs, unplayable. -> Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Worked perfectly, no complaints. -> Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Controls are confusing sometimes. -> Positive
